# AutoLyrics — Whisper Fine-Tuning

> Fine-tune OpenAI Whisper-small on the `gmenon/slt-lyrics-audio` dataset using LoRA (PEFT).

---

In [1]:
# ================================
# 1. Install required packages (run once)
# ================================
!pip install -U torchao jiwer transformers datasets peft librosa openai-whisper accelerate pyloudnorm

In [2]:
# ================================
# 2. Imports and constants
# ================================
import os
import re
import json
import copy
import gc
import random
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
import librosa
import pyloudnorm as pyln
import whisper
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from datasets import load_dataset
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from jiwer import wer, cer
from difflib import SequenceMatcher

# Constants
MODEL_NAME = "openai/whisper-small"  # Base transformer model
DATASET_NAME = "gmenon/slt-lyrics-audio"
OUTPUT_DIR = "./whisper-lora-autolyrics"
LANGUAGE = "en"
TASK = "transcribe"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

TARGET_SR = 16000
MAX_DURATION = 30.0
MIN_DURATION = 2.0
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
BATCH_SIZE = 4
MAX_LABEL_TOKENS = 448
SEED = 42

# Training hyperparameters
LR = 2e-5
NUM_EPOCHS = 2
EVAL_EVERY = 1
N_VAL_SAMPLES = 50
BEAM_SIZE = 1                     # greedy decoding – safer for fine-tuning
PATIENCE = 3
GRAD_ACCUM_STEPS = 2
# MAX_NEW_TOKENS will be set dynamically after dataset creation

SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
# ================================
# 3. Load dataset from Hugging Face
# ================================
print(f"Loading dataset: {DATASET_NAME} ...")
# gmenon/slt-lyrics-audio has a single 'train' split
hf_dataset = load_dataset(DATASET_NAME, split="train", trust_remote_code=True)
print(f"Dataset loaded. Columns: {hf_dataset.column_names}")
print(f"Total examples: {len(hf_dataset)}")
# Inspect one example to confirm column names
print("Sample keys:", list(hf_dataset[0].keys()))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'gmenon/slt-lyrics-audio' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'gmenon/slt-lyrics-audio' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading dataset: gmenon/slt-lyrics-audio ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/641 [00:00<?, ?B/s]

data/train-00000-of-00012-bdae940d202447(…):   0%|          | 0.00/416M [00:00<?, ?B/s]

...

data/train-00011-of-00012-09e896ff3287f3(…):   0%|          | 0.00/420M [00:00<?, ?B/s]

data/eval-00000-of-00001-023ba0c672c3d02(…):   0%|          | 0.00/277M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9538 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/507 [00:00<?, ? examples/s]

Dataset loaded. Columns: ['audio', 'transcription']
Total examples: 9538
Sample keys: ['audio', 'transcription']


In [4]:
# ================================
# 4. Helper functions
# ================================

# ------------------------------------------------------------------
# Audio preprocessing constants
# ------------------------------------------------------------------
LUFS_TARGET   = -23.0   # EBU R128 broadcast loudness target
LUFS_HEADROOM =  1.0    # dB of headroom to prevent clipping after normalisation
SILENCE_TOP_DB = 30     # librosa.effects.trim threshold (higher = more aggressive)


def _remove_dc_offset(waveform: torch.Tensor) -> torch.Tensor:
    """Subtract the mean so the waveform is centred at zero."""
    return waveform - waveform.mean()


def _trim_silence(waveform: torch.Tensor) -> torch.Tensor:
    """Strip leading/trailing silence using librosa, return a tensor."""
    arr = waveform.numpy()
    trimmed, _ = librosa.effects.trim(arr, top_db=SILENCE_TOP_DB)
    return torch.from_numpy(trimmed)


def _loudness_normalize(waveform: torch.Tensor, sr: int) -> torch.Tensor:
    """
    Loudness-normalise to LUFS_TARGET (integrated loudness, EBU R128).
    Falls back to peak normalisation if the clip is too short / silent for
    the meter to produce a valid reading.
    """
    arr = waveform.numpy().astype("float64")
    meter = pyln.Meter(sr)                          # BS.1770 meter
    loudness = meter.integrated_loudness(arr)

    if not (loudness > -70.0):                      # ungated / silent clip
        peak = arr.max() if arr.max() != 0 else 1.0
        arr = arr / peak
    else:
        arr = pyln.normalize.loudness(arr, loudness, LUFS_TARGET)
        # Hard-clip headroom guard
        limit = 10 ** (-LUFS_HEADROOM / 20.0)
        arr = arr.clip(-limit, limit)

    return torch.from_numpy(arr.astype("float32"))


def _preprocess_waveform(waveform: torch.Tensor, sr: int) -> torch.Tensor:
    """
    Full preprocessing chain applied to every waveform after resampling:
      1. DC offset removal
      2. Silence trimming
      3. Loudness normalisation (LUFS / peak fallback)
    Returns a 1-D float32 tensor.
    """
    waveform = _remove_dc_offset(waveform)
    waveform = _trim_silence(waveform)
    if len(waveform) == 0:          # edge-case: fully silent after trimming
        return waveform
    waveform = _loudness_normalize(waveform, sr)
    return waveform


def load_audio_array(audio_array, orig_sr):
    """Convert HF audio dict to a preprocessed mono float32 waveform tensor at TARGET_SR."""
    waveform = torch.tensor(audio_array, dtype=torch.float32)
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)          # (1, T)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if orig_sr != TARGET_SR:
        waveform = T.Resample(orig_sr, TARGET_SR)(waveform)
    waveform = waveform.squeeze(0)                # (T,)
    waveform = _preprocess_waveform(waveform, TARGET_SR)
    return waveform


def load_audio(wav_path: str):
    """Load a wav/mp3 file from disk (kept for test-set reload compatibility)."""
    waveform, sr = torchaudio.load(wav_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        waveform = T.Resample(sr, TARGET_SR)(waveform)
    waveform = waveform.squeeze(0)
    waveform = _preprocess_waveform(waveform, TARGET_SR)
    return waveform


# ------------------------------------------------------------------
# Text normalisation
# ------------------------------------------------------------------
_BRACKET_RE  = re.compile(r"[\[\(][^\]\)]*[\]\)]")  # (chorus), [verse 1], etc.
_MULTI_SPACE = re.compile(r" +")
_STRIP_PUNCT = re.compile(r"[^a-z0-9\s'\-]")        # keep apostrophe and hyphen


def normalize_text(text: str) -> str:
    """
    Normalise a lyrics string so label tokens are consistent:
      1. Remove structural annotations like (chorus) / [verse 2]
      2. Lowercase
      3. Normalise unicode apostrophes / dashes to ASCII
      4. Strip punctuation except apostrophe and hyphen
      5. Collapse multiple spaces and strip
    """
    text = _BRACKET_RE.sub(" ", text)                    # drop annotations
    text = text.lower()
    text = text.replace('’', "'").replace('‘', "'")  # curly apostrophes
    text = text.replace('—', '-').replace('–', '-')    # em/en dash
    text = _STRIP_PUNCT.sub(" ", text)                   # remove other punctuation
    text = _MULTI_SPACE.sub(" ", text).strip()
    return text


In [6]:
import pyloudnorm
# ================================
# 5. Parse gmenon/slt-lyrics-audio
# ================================
def parse_slt_lyrics(hf_ds):
    """
    Convert gmenon/slt-lyrics-audio rows into the sample dicts the rest
    of the pipeline expects.

    Expected HF columns (adjust if the dataset columns differ):
      audio  -> dict with keys 'array' (np.ndarray) and 'sampling_rate'
      text   -> ground-truth lyric string
      singer -> singer / track identifier (used for train/val/test split)

    If the dataset uses different column names, update the key lookups below.
    """
    # Ensure audio column is decoded
    hf_ds = hf_ds.cast_column("audio", hf_ds.features["audio"])

    samples = []
    # Identify the lyrics column (try common names)
    text_col  = next((c for c in ["text", "lyrics", "transcript", "transcription", "sentence"] if c in hf_ds.column_names), None)
    singer_col = next((c for c in ["singer", "speaker_id", "speaker", "artist"] if c in hf_ds.column_names), None)

    if text_col is None:
        raise ValueError(f"Cannot find lyrics column. Available: {hf_ds.column_names}")

    for i, row in enumerate(tqdm(hf_ds, desc="Parsing slt-lyrics-audio")):
        audio_info = row["audio"]
        transcript = normalize_text(row[text_col])
        if not transcript:
            continue

        orig_sr = audio_info["sampling_rate"]

        try:
            waveform = load_audio_array(audio_info["array"], orig_sr)
        except ValueError as e:
            # Catch errors specifically related to audio length required by pyloudnorm
            if "Audio must have length greater than the block size" in str(e):
                print(f"Skipping sample {i} due to audio being too short for loudness normalization: {e}")
                continue
            else:
                raise # Re-raise other ValueErrors

        duration = len(waveform) / TARGET_SR

        if duration < MIN_DURATION or duration > MAX_DURATION:
            continue

        # Use singer column if present, else bucket by index for splitting
        singer = str(row[singer_col]) if singer_col else f"spk_{i % 20:03d}"

        samples.append({
            "audio_array": audio_info["array"],   # kept in memory (small dataset)
            "audio_sr":    orig_sr,
            "transcript":  transcript,
            "source":      "slt_lyrics",
            "singer":      singer,
            "start_sec":   0.0,
            "end_sec":     duration,
        })

    print(f"[slt-lyrics-audio] Loaded {len(samples)} usable samples.")
    return samples

print("Parsing dataset...")
all_samples = parse_slt_lyrics(hf_dataset)
print(f"Total raw samples: {len(all_samples)}")

if len(all_samples) == 0:
    raise RuntimeError("No samples found! Check dataset columns and duration filters.")


Parsing dataset...


Parsing slt-lyrics-audio:   1%|          | 72/9538 [00:02<06:18, 25.03it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:   1%|          | 75/9538 [00:02<06:13, 25.34it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:   1%|          | 78/9538 [00:02<06:11, 25.46it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:   1%|          | 108/9538 [00:03<04:34, 34.31it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt

Skipping sample 2512 due to audio being too short for loudness normalization: Audio must have length greater than the block size.


Parsing slt-lyrics-audio:  27%|██▋       | 2595/9538 [01:10<02:57, 39.14it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:  27%|██▋       | 2599/9538 [01:10<03:04, 37.53it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:  28%|██▊       | 2666/9538 [01:12<02:48, 40.72it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:  28%|██▊       | 2676/9538 [01:12<02:44, 41.73it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Pars

Skipping sample 3525 due to audio being too short for loudness normalization: Audio must have length greater than the block size.


...

Skipping sample 8552 due to audio being too short for loudness normalization: Audio must have length greater than the block size.


Parsing slt-lyrics-audio:  90%|████████▉ | 8572/9538 [03:52<00:25, 37.29it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:  90%|█████████ | 8625/9538 [03:54<00:38, 23.82it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:  90%|█████████ | 8628/9538 [03:54<00:37, 24.06it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:  91%|█████████ | 8695/9538 [03:56<00:20, 40.64it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Pars

Skipping sample 9389 due to audio being too short for loudness normalization: Audio must have length greater than the block size.


Parsing slt-lyrics-audio:  99%|█████████▊| 9409/9538 [04:15<00:03, 40.62it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio:  99%|█████████▉| 9458/9538 [04:16<00:01, 40.31it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio: 100%|█████████▉| 9496/9538 [04:17<00:01, 40.48it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Parsing slt-lyrics-audio: 100%|█████████▉| 9523/9538 [04:18<00:00, 27.83it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Pars

[slt-lyrics-audio] Loaded 8985 usable samples.
Total raw samples: 8985


In [7]:
# ================================
# 6. Build train / val / test splits
# ================================
def split_dataset(all_samples):
    singers = {s["singer"] for s in all_samples}
    singer_list = list(singers)
    random.shuffle(singer_list)
    n_train = int(len(singer_list) * TRAIN_RATIO)
    n_val   = int(len(singer_list) * VAL_RATIO)
    train_singers = set(singer_list[:n_train])
    val_singers   = set(singer_list[n_train:n_train + n_val])
    test_singers  = set(singer_list[n_train + n_val:])

    train = [s for s in all_samples if s["singer"] in train_singers]
    val   = [s for s in all_samples if s["singer"] in val_singers]
    test  = [s for s in all_samples if s["singer"] in test_singers]
    return train, val, test


class LyricsDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples   = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        waveform = load_audio_array(s["audio_array"], s["audio_sr"])

        max_samples = int(MAX_DURATION * TARGET_SR)
        if len(waveform) < max_samples:
            waveform = torch.nn.functional.pad(waveform, (0, max_samples - len(waveform)))
        else:
            waveform = waveform[:max_samples]

        features = self.processor.feature_extractor(
            waveform.numpy(), sampling_rate=TARGET_SR, return_tensors="pt"
        ).input_features[0]

        labels = self.processor.tokenizer(
            s["transcript"], add_special_tokens=True
        ).input_ids[:MAX_LABEL_TOKENS]

        return {
            "input_features": features,
            "labels":         torch.tensor(labels, dtype=torch.long),
            "transcript":     s["transcript"],
        }


def collate_fn(batch):
    input_features = torch.stack([b["input_features"] for b in batch])
    max_len = max(len(b["labels"]) for b in batch)
    labels  = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        labels[i, :len(b["labels"])] = b["labels"]
    return {
        "input_features": input_features,
        "labels":         labels,
        "transcript":     [b["transcript"] for b in batch],
    }


train_data, val_data, test_data = split_dataset(all_samples)
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# Load processor
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

# Create datasets
train_dataset = LyricsDataset(train_data, processor)
val_dataset   = LyricsDataset(val_data,   processor)
test_dataset  = LyricsDataset(test_data,  processor)

# Dynamically set MAX_NEW_TOKENS
all_label_lengths = [
    len(processor.tokenizer(s["transcript"], add_special_tokens=True).input_ids)
    for s in train_data
]
MAX_NEW_TOKENS = int(np.percentile(all_label_lengths, 99)) + 10
print(f"Setting MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")

Train: 6286, Val: 1341, Test: 1358


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Setting MAX_NEW_TOKENS = 28


In [8]:
# ================================
# 7. Load base model and apply LoRA
# ================================
def load_base_model():
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=MODEL_DTYPE
    ).to(DEVICE)
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    return model

def apply_lora(model, target_modules=None):
    if target_modules is None:
        module_names = set()
        for name, _ in model.named_modules():
            parts = name.split(".")
            if parts[-1] in ("q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"):
                module_names.add(parts[-1])
        target_modules = sorted(module_names)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        target_modules=target_modules,
    )
    model = get_peft_model(model, lora_config)
    return model

print("Loading base Whisper model...")
base_model = load_base_model()
print("Applying LoRA (encoder+decoder)...")
model = apply_lora(base_model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total params: {total/1e6:.1f}M, Trainable: {trainable/1e6:.3f}M ({100*trainable/total:.2f}%)")

Loading base Whisper model...


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Applying LoRA (encoder+decoder)...
Total params: 245.0M, Trainable: 3.244M (1.32%)


In [9]:
# ================================
# 8. Load base model and apply LoRA
# ================================
def load_base_model():
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=MODEL_DTYPE
    ).to(DEVICE)
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    return model

def apply_lora(model, target_modules=None):
    if target_modules is None:
        module_names = set()
        for name, _ in model.named_modules():
            parts = name.split(".")
            if parts[-1] in ("q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"):
                module_names.add(parts[-1])
        target_modules = sorted(module_names)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=8,                # reduced from 32 for stability
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        target_modules=target_modules,
    )
    model = get_peft_model(model, lora_config)
    return model

print("Loading base Whisper model...")
base_model = load_base_model()
print("Applying LoRA (encoder+decoder)...")
model = apply_lora(base_model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total/1e6:.1f}M, Trainable: {trainable/1e6:.3f}M ({100*trainable/total:.2f}%)")

Loading base Whisper model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Applying LoRA (encoder+decoder)...
Total params: 245.0M, Trainable: 3.244M (1.32%)


In [10]:
# ================================
# 9. Evaluation function (fixed for PEFT)
# ================================
def evaluate(eval_model, dataset, label="val", n_samples=None):
    if n_samples:
        dataset = Subset(dataset, range(min(n_samples, len(dataset))))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    total_loss = 0.0
    all_preds = []
    all_refs = []

    # Unwrap to the inner WhisperForConditionalGeneration for both loss and generate().
    # Calling .eval() on the inner model disables dropout so val metrics are deterministic;
    # we restore training mode afterwards if the caller was training.
    if hasattr(eval_model, "base_model"):
        gen_model = eval_model.base_model.model
    else:
        gen_model = eval_model

    was_training = gen_model.training
    gen_model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  eval [{label}]", leave=False):
            feats = batch["input_features"].to(DEVICE, dtype=MODEL_DTYPE)
            labels = batch["labels"].to(DEVICE)

            loss = gen_model(input_features=feats, labels=labels).loss  # FIX: use unwrapped model — PEFT wrapper conflicts with input_features kwarg
            total_loss += loss.item()

            generated_ids = gen_model.generate(
                input_features=feats,
                num_beams=BEAM_SIZE,
                max_new_tokens=MAX_NEW_TOKENS,
                language="en",
                task="transcribe",
                suppress_tokens=[],
                forced_decoder_ids=None,
            )
            preds = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            all_preds.extend(preds)
            all_refs.extend(batch["transcript"])

    if was_training:
        gen_model.train()

    avg_loss = total_loss / len(loader)
    pairs = [(r.strip(), p.strip()) for r, p in zip(all_refs, all_preds) if r.strip() and p.strip()]
    if not pairs:
        return avg_loss, 1.0, 1.0
    refs = [p[0] for p in pairs]
    hyps = [p[1] for p in pairs]
    avg_wer = wer(refs, hyps)
    avg_cer = cer(refs, hyps)
    return avg_loss, avg_wer, avg_cer

In [11]:
# ================================
# 10. Training loop with gradient accumulation
# ================================
def train_model(peft_model, experiment_name, ckpt_dir):
    # Renamed parameter: peft_model (was train_model — shadowed the function name)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, peft_model.parameters()), lr=LR
    )
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
    num_steps = NUM_EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*num_steps), num_training_steps=num_steps)

    peft_model.train()
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    print(f"\n{'='*60}\nTraining: {experiment_name}")
    print(f"Epochs={NUM_EPOCHS}, LR={LR}, Batch={BATCH_SIZE}, GradAccum={GRAD_ACCUM_STEPS}")
    print(f"{'='*60}")

    # Unwrap once outside the epoch loop — avoids repeated attribute lookups and
    # sidesteps the PEFT Seq2SeqLM wrapper's conflict with Whisper's input_features kwarg.
    # Gradients still flow through LoRA layers since they live inside inner_model.
    inner_model = peft_model.base_model.model

    global_step = 0
    for epoch in range(1, NUM_EPOCHS+1):
        epoch_loss = 0.0
        optimizer.zero_grad()
        for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")):
            feats = batch["input_features"].to(DEVICE, dtype=MODEL_DTYPE)
            labels = batch["labels"].to(DEVICE)
            loss = inner_model(input_features=feats, labels=labels).loss
            loss = loss / GRAD_ACCUM_STEPS
            loss.backward()
            epoch_loss += loss.item() * GRAD_ACCUM_STEPS

            if (i+1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            global_step += 1

        # Flush any remaining gradients from the last partial accumulation window
        if (i+1) % GRAD_ACCUM_STEPS != 0:
            torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        avg_train_loss = epoch_loss / len(train_loader)

        if epoch % EVAL_EVERY == 0:
            val_loss, val_wer, val_cer = evaluate(peft_model, val_dataset, label="val", n_samples=N_VAL_SAMPLES)
            history.append({
                "epoch": epoch,
                "train_loss": avg_train_loss,
                "val_loss": val_loss,
                "val_wer": val_wer,
                "val_cer": val_cer
            })
            print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}, val_loss={val_loss:.4f}, WER={val_wer*100:.2f}%, CER={val_cer*100:.2f}%")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy({k: v.cpu() for k, v in peft_model.state_dict().items() if "lora" in k})
                patience_counter = 0
                print(f"  ✓ New best val_loss={val_loss:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"Early stopping at epoch {epoch}")
                    break

    if best_state is not None:
        current = peft_model.state_dict()
        for k, v in best_state.items():
            if k in current:
                current[k] = v.to(DEVICE)
        peft_model.load_state_dict(current)
        print(f"Restored best model with val_loss={best_val_loss:.4f}")

    peft_model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    return history

In [12]:
# ================================
# 11. Run experiments
# ================================
print("\nEXPERIMENT 1: Zero-shot baseline")
zs_loss, zs_wer, zs_cer = evaluate(model, val_dataset, label="zero-shot", n_samples=N_VAL_SAMPLES)
print(f"Zero-shot: Loss={zs_loss:.4f}, WER={zs_wer*100:.2f}%, CER={zs_cer*100:.2f}%")

print("\nEXPERIMENT 2: LoRA fine-tuning (encoder+decoder)")
history = train_model(model, "LoRA Enc+Dec", f"{SAVE_DIR}/lora_best")

final_loss, final_wer, final_cer = evaluate(model, val_dataset, label="final", n_samples=N_VAL_SAMPLES)
print(f"\nFinal validation: WER={final_wer*100:.2f}% (improvement from {zs_wer*100:.2f}%)")

results = {
    "zero_shot": {"wer": zs_wer, "cer": zs_cer, "loss": zs_loss},
    "lora_best": {"wer": final_wer, "cer": final_cer, "loss": final_loss, "history": history}
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {SAVE_DIR}/results.json")


EXPERIMENT 1: Zero-shot baseline


  eval [zero-shot]:   0%|          | 0/13 [00:00<?, ?it/s][transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=28) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressToken

Zero-shot: Loss=6.5030, WER=74.32%, CER=49.39%

EXPERIMENT 2: LoRA fine-tuning (encoder+decoder)

Training: LoRA Enc+Dec
Epochs=2, LR=2e-05, Batch=4, GradAccum=2


Epoch 1/2:   0%|          | 0/1572 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Epoch 1/2:   0%|          | 3/1572 [00:02<21:12,  1.23it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Epoch 1/2:   1%|          | 11/1572 [00:07<15:55,  1.63it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Epoch 1/2:   1%|          | 17/1572 [00:11<15:54,  1.63it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Epoch 1/2:   2%|▏         | 25/1572 [00:16<15:49,  1.63it/s]/usr/local/lib/python3

Epoch 1: train_loss=2.5629, val_loss=1.1157, WER=56.42%, CER=37.46%
  ✓ New best val_loss=1.1157


Epoch 2/2:   1%|          | 8/1572 [00:05<17:21,  1.50it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Epoch 2/2:   1%|▏         | 20/1572 [00:13<16:51,  1.53it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Epoch 2/2:   1%|▏         | 21/1572 [00:14<16:41,  1.55it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Epoch 2/2:   2%|▏         | 39/1572 [00:25<16:30,  1.55it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Epoch 2/2:   3%|▎         | 40/1572 [00:26<16:41,  1.53it/s]/usr/local/li

Epoch 2: train_loss=1.1725, val_loss=1.0716, WER=58.45%, CER=39.61%
  ✓ New best val_loss=1.0716
Restored best model with val_loss=1.0716


  eval [final]:  92%|█████████▏| 12/13 [00:10<00:00,  1.13it/s][transformers] Both `max_new_tokens` (=28) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
                                                               


Final validation: WER=58.45% (improvement from 74.32%)
Results saved to ./checkpoints/results.json


In [13]:
# ================================
# 12. Evaluation on test set
# ================================
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

base_test = load_base_model()
best_model = PeftModel.from_pretrained(base_test, f"{SAVE_DIR}/lora_best").to(DEVICE)

test_loss, test_wer, test_cer = evaluate(best_model, test_dataset, label="test")
print(f"Test set: WER={test_wer*100:.2f}%, CER={test_cer*100:.2f}%, Loss={test_loss:.4f}")

improvement = (zs_wer - test_wer) / zs_wer * 100
print(f"Relative improvement over zero-shot: {improvement:.1f}%")
if improvement >= 15:
    print("✓ Target achieved: >15% WER reduction")
else:
    print("✗ Target not yet reached – consider more data or larger model")

test_results = {
    "wer": test_wer,
    "cer": test_cer,
    "loss": test_loss,
    "improvement_percent": improvement,
}
with open(f"{SAVE_DIR}/test_results.json", "w") as f:
    json.dump(test_results, f, indent=2)
print("\nAll done!")


TEST SET EVALUATION


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  eval [test]:   3%|▎         | 11/340 [00:09<04:37,  1.18it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
[transformers] Both `max_new_tokens` (=28) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  eval [test]:   9%|▊         | 29/340 [00:27<04:57,  1.05it/s]/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
[transformers] Both `max_new_tokens` (=28) and `max_length`(=448) seem to have been set. `max_

Test set: WER=60.52%, CER=41.29%, Loss=1.1426
Relative improvement over zero-shot: 18.6%
✓ Target achieved: >15% WER reduction

All done!


In [15]:
# ================================
# 13. Performance Report
# ================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from difflib import SequenceMatcher
from jiwer import wer, cer
from torch.utils.data import DataLoader, Subset
import json, os, datetime

REPORT_DIR = "./report"
os.makedirs(REPORT_DIR, exist_ok=True)

# ── 13a. Reload saved results ──────────────────────────────────────────────────
with open(f"{SAVE_DIR}/results.json") as f:
    results = json.load(f)
with open(f"{SAVE_DIR}/test_results.json") as f:
    test_results = json.load(f)

history     = results["lora_best"]["history"]   # list of epoch dicts
zs_wer      = results["zero_shot"]["wer"]
zs_cer      = results["zero_shot"]["cer"]
zs_loss     = results["zero_shot"]["loss"]
final_wer   = results["lora_best"]["wer"]
final_cer   = results["lora_best"]["cer"]
final_loss  = results["lora_best"]["loss"]
test_wer    = test_results["wer"]
test_cer    = test_results["cer"]
test_loss   = test_results["loss"]
improvement = test_results["improvement_percent"]

# ── 13b. Qualitative: per-sample predictions on test set ──────────────────────
N_QUAL = 20   # number of test samples to decode for the qualitative table
qual_loader = DataLoader(
    Subset(test_dataset, range(min(N_QUAL, len(test_dataset)))),
    batch_size=4, shuffle=False, collate_fn=collate_fn
)

qual_rows = []   # (reference, hypothesis, wer, cer, similarity)

inner = best_model.base_model.model if hasattr(best_model, "base_model") else best_model
inner.eval()
with torch.no_grad():
    for batch in qual_loader:
        feats = batch["input_features"].to(DEVICE, dtype=MODEL_DTYPE)
        gen_ids = inner.generate(
            input_features=feats,
            num_beams=BEAM_SIZE,
            max_new_tokens=MAX_NEW_TOKENS,
            language="en",
            task="transcribe",
            suppress_tokens=[],
            forced_decoder_ids=None,
        )
        preds = processor.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        for ref, hyp in zip(batch["transcript"], preds):
            ref, hyp = ref.strip(), hyp.strip()
            if not ref:
                continue
            sample_wer = wer(ref, hyp) if hyp else 1.0
            sample_cer = cer(ref, hyp) if hyp else 1.0
            sim = SequenceMatcher(None, ref.lower(), hyp.lower()).ratio()
            qual_rows.append((ref, hyp, sample_wer, sample_cer, sim))

# ── 13c. Plot: Training curves + metric comparison ────────────────────────────
epochs_list  = [h["epoch"]      for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses   = [h["val_loss"]   for h in history]
val_wers     = [h["val_wer"]    for h in history]
val_cers     = [h["val_cer"]    for h in history]

fig = plt.figure(figsize=(18, 14))
fig.suptitle("AutoLyrics — Whisper-small + LoRA Performance Report", fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Loss curves
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(epochs_list, train_losses, "o-", label="Train Loss", color="#2563EB", linewidth=2)
ax1.plot(epochs_list, val_losses,   "s--", label="Val Loss",   color="#DC2626", linewidth=2)
ax1.axhline(zs_loss, color="gray", linestyle=":", label=f"Zero-shot Loss ({zs_loss:.3f})")
ax1.set_title("Training & Validation Loss per Epoch")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_xticks(epochs_list)

# 2. WER / CER curves
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(epochs_list, [w*100 for w in val_wers], "o-", label="WER %", color="#7C3AED", linewidth=2)
ax2.plot(epochs_list, [c*100 for c in val_cers], "s--", label="CER %", color="#D97706", linewidth=2)
ax2.axhline(zs_wer*100, color="#7C3AED", linestyle=":", alpha=0.5, label=f"ZS-WER {zs_wer*100:.1f}%")
ax2.axhline(zs_cer*100, color="#D97706", linestyle=":", alpha=0.5, label=f"ZS-CER {zs_cer*100:.1f}%")
ax2.set_title("Val WER & CER per Epoch")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("% Error")
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)
ax2.set_xticks(epochs_list)

# 3. Bar chart — WER comparison
ax3 = fig.add_subplot(gs[1, 0])
stages = ["Zero-shot", "Best Val", "Test Set"]
wers   = [zs_wer*100, final_wer*100, test_wer*100]
colors = ["#94A3B8", "#3B82F6", "#10B981"]
bars = ax3.bar(stages, wers, color=colors, edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, wers):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.2f}%",
             ha="center", va="bottom", fontsize=9, fontweight="bold")
ax3.set_title("WER Comparison"); ax3.set_ylabel("WER (%)"); ax3.grid(axis="y", alpha=0.3)

# 4. Bar chart — CER comparison
ax4 = fig.add_subplot(gs[1, 1])
cers = [zs_cer*100, final_cer*100, test_cer*100]
bars2 = ax4.bar(stages, cers, color=colors, edgecolor="white", linewidth=1.2)
for bar, val in zip(bars2, cers):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.2f}%",
             ha="center", va="bottom", fontsize=9, fontweight="bold")
ax4.set_title("CER Comparison"); ax4.set_ylabel("CER (%)"); ax4.grid(axis="y", alpha=0.3)

# 5. Per-sample WER distribution (histogram)
ax5 = fig.add_subplot(gs[1, 2])
sample_wers = [r[2]*100 for r in qual_rows]
ax5.hist(sample_wers, bins=10, color="#6366F1", edgecolor="white", alpha=0.85)
ax5.axvline(test_wer*100, color="red", linestyle="--", label=f"Avg WER {test_wer*100:.1f}%")
ax5.set_title("Per-sample WER Distribution\n(Qualitative subset)")
ax5.set_xlabel("WER (%)"); ax5.set_ylabel("Count")
ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

# 6. Similarity score distribution
ax6 = fig.add_subplot(gs[2, 0])
sims = [r[4]*100 for r in qual_rows]
ax6.hist(sims, bins=10, color="#059669", edgecolor="white", alpha=0.85)
ax6.axvline(float(sum(sims))/len(sims), color="red", linestyle="--",
            label=f"Mean {sum(sims)/len(sims):.1f}%")
ax6.set_title("Sequence Similarity Score\n(Qualitative subset)")
ax6.set_xlabel("Similarity (%)"); ax6.set_ylabel("Count")
ax6.legend(fontsize=8); ax6.grid(alpha=0.3)

# 7. Summary stats table
ax7 = fig.add_subplot(gs[2, 1:])
ax7.axis("off")
table_data = [
    ["Metric",              "Zero-shot",          "Best Val",           "Test Set"],
    ["WER (%)",             f"{zs_wer*100:.2f}",  f"{final_wer*100:.2f}", f"{test_wer*100:.2f}"],
    ["CER (%)",             f"{zs_cer*100:.2f}",  f"{final_cer*100:.2f}", f"{test_cer*100:.2f}"],
    ["Loss",                f"{zs_loss:.4f}",     f"{final_loss:.4f}",    f"{test_loss:.4f}"],
    ["WER Improvement (%)", "—",                  f"{(zs_wer-final_wer)/zs_wer*100:.1f}",
                                                   f"{improvement:.1f}"],
    ["Epochs trained",      "—",                  f"{NUM_EPOCHS}",        "—"],
    ["LoRA rank (r)",       "—",                  "8",                    "—"],
    ["Learning rate",       "—",                  f"{LR}",                "—"],
    ["Batch size",          "—",                  f"{BATCH_SIZE}",        "—"],
    ["Grad accum steps",    "—",                  f"{GRAD_ACCUM_STEPS}",  "—"],
]
tbl = ax7.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#1E3A5F"); cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#EFF6FF")
ax7.set_title("Summary Statistics", fontweight="bold", pad=12)

plt.savefig(f"{REPORT_DIR}/performance_report.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✓ Plot saved → {REPORT_DIR}/performance_report.png")

# ── 13d. Qualitative predictions table (printed + saved to txt) ───────────────
qual_lines = [
    "=" * 90,
    "QUALITATIVE SAMPLE PREDICTIONS",
    f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "=" * 90,
]
for i, (ref, hyp, sw, sc, sim) in enumerate(qual_rows, 1):
    qual_lines += [
        f"\n[Sample {i:02d}]",
        f"  REF : {ref}",
        f"  HYP : {hyp}",
        f"  WER : {sw*100:.1f}%   CER : {sc*100:.1f}%   Similarity : {sim*100:.1f}%",
    ]
qual_text = "\n".join(qual_lines)
print(qual_text)
with open(f"{REPORT_DIR}/qualitative_predictions.txt", "w") as f:
    f.write(qual_text)
print(f"\n✓ Qualitative predictions saved → {REPORT_DIR}/qualitative_predictions.txt")

# ── 13e. Machine-readable summary JSON ────────────────────────────────────────
report_summary = {
    "generated_at": datetime.datetime.now().isoformat(),
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "num_epochs": NUM_EPOCHS,
    "lora_rank": 8,
    "lora_alpha": 16,
    "learning_rate": LR,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "zero_shot": {"wer": zs_wer, "cer": zs_cer, "loss": zs_loss},
    "best_val":  {"wer": final_wer, "cer": final_cer, "loss": final_loss},
    "test":      {"wer": test_wer,  "cer": test_cer,  "loss": test_loss,
                  "wer_improvement_pct": improvement},
    "training_history": history,
    "target_achieved_15pct_wer_reduction": improvement >= 15,
}
with open(f"{REPORT_DIR}/report_summary.json", "w") as f:
    json.dump(report_summary, f, indent=2)
print(f"✓ Summary JSON saved  → {REPORT_DIR}/report_summary.json")

# ── 13f. Final console summary ────────────────────────────────────────────────
print("\n" + "="*60)
print("  PERFORMANCE REPORT SUMMARY")
print("="*60)
print(f"  Zero-shot  WER : {zs_wer*100:.2f}%   CER : {zs_cer*100:.2f}%   Loss : {zs_loss:.4f}")
print(f"  Best Val   WER : {final_wer*100:.2f}%   CER : {final_cer*100:.2f}%   Loss : {final_loss:.4f}")
print(f"  Test Set   WER : {test_wer*100:.2f}%   CER : {test_cer*100:.2f}%   Loss : {test_loss:.4f}")
print(f"  WER improvement over zero-shot : {improvement:.1f}%")
print(f"  Target (>15% WER reduction)    : {'✓ ACHIEVED' if improvement >= 15 else '✗ Not yet reached'}")
print("="*60)
print(f"\nAll report files written to: {REPORT_DIR}/")
print("  • performance_report.png")
print("  • qualitative_predictions.txt")
print("  • report_summary.json")

[transformers] Both `max_new_tokens` (=28) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=28) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=28) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=28) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

✓ Plot saved → ./report/performance_report.png
QUALITATIVE SAMPLE PREDICTIONS
Generated: 2026-05-18 14:58:04

[Sample 01]
  REF : and you've seen it all before but the wolf's outside your door
  HYP : can you see it before but the wolfs are such a tool
  WER : 75.0%   CER : 38.7%   Similarity : 72.6%

[Sample 02]
  REF : and talk is cheap awhen the story is good
  HYP : and come to see when the story is through
  WER : 55.6%   CER : 41.5%   Similarity : 65.9%

[Sample 03]
  REF : something more than dreams to
  HYP : something more than dreams to
  WER : 0.0%   CER : 0.0%   Similarity : 100.0%

[Sample 04]
  REF : but it ain't a bit of use
  HYP : try to be the biggest
  WER : 100.0%   CER : 72.0%   Similarity : 26.1%

[Sample 05]
  REF : so what can you say
  HYP : i'm not gonna let you
  WER : 100.0%   CER : 84.2%   Similarity : 40.0%

[Sample 06]
  REF : only one demand
  HYP : i see your own death
  WER : 166.7%   CER : 93.3%   Similarity : 45.7%

[Sample 07]
  REF : a sulley roit 